# 02 — LTV Prediction

Leakage-aware LTV modeling and business profitability simulation.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Objective

Build LTV models and compare snapshot vs leakage-aware feature sets.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
SYNTHETIC_DIR = Path('../data/synthetic')
REPORTS_DIR = Path('../reports')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
wallet_path = RAW_DIR / 'digital_wallet_ltv_dataset.csv'
if not wallet_path.exists():
    raise FileNotFoundError('Put digital_wallet_ltv_dataset.csv under data/raw/')
df = pd.read_csv(PROCESSED_DIR / 'wallet_retention_baseline.csv') if (PROCESSED_DIR / 'wallet_retention_baseline.csv').exists() else pd.read_csv(wallet_path)
print(df.shape)
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance


## Leakage analysis


In [ ]:
target='LTV'
corr = df.select_dtypes(include=['number']).corr(numeric_only=True)
if target in corr.columns:
    display(corr[target].drop(target).sort_values(key=abs, ascending=False).to_frame('correlation_with_LTV'))
possible_leakage = ['Total_Spent','Loyalty_Points_Earned','Cashback_Received','rfm_score','monetary_score','ltv_quartile']
print('Potential leakage variables present:', [c for c in possible_leakage if c in df.columns])


## Feature sets


In [ ]:
base_drop=['Customer_ID','LTV']
snapshot_features=[c for c in df.columns if c not in base_drop]
leakage_candidates=['Total_Spent','Loyalty_Points_Earned','Cashback_Received','rfm_score','monetary_score','ltv_quartile']
leakage_aware_features=[c for c in snapshot_features if c not in leakage_candidates]
print('Snapshot:', len(snapshot_features), 'Leakage-aware:', len(leakage_aware_features))


In [ ]:
def rmse(y, p): return mean_squared_error(y, p, squared=False)
def make_preprocessor(X):
    num=X.select_dtypes(include=['int64','float64']).columns.tolist()
    cat=X.select_dtypes(include=['object','category']).columns.tolist()
    return ColumnTransformer([('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]), num), ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat)])
def evaluate(y,p): return {'MAE':mean_absolute_error(y,p),'RMSE':rmse(y,p),'R2':r2_score(y,p)}


## Model comparison


In [ ]:
def run_experiment(feature_cols, experiment_name):
    X=df[feature_cols]; y=df[target]
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE)
    prep=make_preprocessor(X_train)
    models={'mean_baseline':None,'linear_regression':LinearRegression(),'ridge':Ridge(alpha=10),'random_forest':RandomForestRegressor(n_estimators=300,max_depth=8,min_samples_leaf=10,random_state=RANDOM_STATE),'gradient_boosting':GradientBoostingRegressor(random_state=RANDOM_STATE)}
    rows=[]; fitted={}
    for name, model in models.items():
        if model is None:
            pred=np.repeat(y_train.mean(), len(y_test)); fitted[name]=None
        else:
            pipe=Pipeline([('preprocess',prep),('model',model)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test); fitted[name]=pipe
        m=evaluate(y_test,pred); m.update({'model':name,'experiment':experiment_name}); rows.append(m)
    return pd.DataFrame(rows).sort_values('RMSE'), fitted, (X_train,X_test,y_train,y_test)
snapshot_results, snapshot_models, snapshot_split = run_experiment(snapshot_features, 'snapshot_all_features')
aware_results, aware_models, aware_split = run_experiment(leakage_aware_features, 'leakage_aware')
results=pd.concat([snapshot_results, aware_results])
display(results.sort_values(['experiment','RMSE']))


## Score customers


In [ ]:
best_name=aware_results.sort_values('RMSE').iloc[0]['model']
best_model=aware_models[best_name]
print('Selected model:', best_name)
df['predicted_ltv']=best_model.predict(df[leakage_aware_features])
plt.figure(figsize=(6,6)); plt.scatter(df['LTV'], df['predicted_ltv'], alpha=0.4); plt.title('Actual vs Predicted LTV'); plt.xlabel('Actual LTV'); plt.ylabel('Predicted LTV'); plt.show()


## Feature importance


In [ ]:
X_train,X_test,y_train,y_test=aware_split
perm=permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, scoring='neg_root_mean_squared_error')
importance=pd.DataFrame({'feature':X_test.columns,'importance':perm.importances_mean}).sort_values('importance', ascending=False)
display(importance.head(15))


## LTV segmentation and CAC simulation


In [ ]:
df['ltv_segment']=pd.qcut(df['predicted_ltv'].rank(method='first'), q=4, labels=['Low','Medium','High','Very High'])
display(df.groupby('ltv_segment').agg(customers=('Customer_ID','count'), avg_predicted_ltv=('predicted_ltv','mean'), avg_actual_ltv=('LTV','mean'), avg_transactions=('Total_Transactions','mean')))
rows=[]
for cac in [10,20,30,40,50,75,100]: rows.append({'CAC':cac,'profitable_rate':(df['predicted_ltv']>cac).mean(),'avg_expected_profit':(df['predicted_ltv']-cac).mean()})
cac_df=pd.DataFrame(rows); display(cac_df)
plt.figure(figsize=(7,4)); plt.plot(cac_df['CAC'], cac_df['avg_expected_profit'], marker='o'); plt.axhline(0, linestyle='--'); plt.title('CAC Sensitivity'); plt.show()


## Save outputs


In [ ]:
out=PROCESSED_DIR/'wallet_ltv_predictions.csv'
df.to_csv(out,index=False)
(REPORTS_DIR/'02_ltv_prediction_summary.md').write_text('# LTV Prediction Summary

Compares snapshot and leakage-aware LTV prediction.
')
print('Saved:',out)


## Discussion prompts

1. Why can Total_Spent be leakage?
2. When is the snapshot model still useful?
3. How do you explain LTV prediction to marketing stakeholders?
